# Pose-cluster PCA -- interactive GMM/HDBSCAN exploration

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) -- install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

Interactive re-exploration of `results/<complex>/pose_clusters.csv`'s PCA
embedding (`pc1`/`pc2`), i.e. the same data behind the static
`results/<complex>/pose_clusters_pca.svg`. This notebook does **not**
refit PCA or redo the anchor-Kabsch alignment / partner RMSF trim --
`scripts/pose_cluster_anchor.py` computes and persists `pc1`/`pc2` into
`pose_clusters.csv` (added 2026-08-24 specifically so this notebook could
reuse the pipeline's own embedding rather than duplicate it), so what you
see here always matches that SVG exactly. Re-run
`workflows/postprocessing/Snakefile`'s `pose_cluster` rule (or
`scripts/pose_cluster_anchor.py` directly) first if you've changed
clustering parameters (`config.yaml`'s `cluster:` block) and want fresh
numbers here.

Clustering approach mirrors
`../../NPF-ab-initio-modelling/ABCfold_NPF_pipeline/scripts/_notebook_setup_functions.py`'s
`plot_pca` (`_fit_gmm_bic_sweep` / `_fit_hdbscan_dbcv_search`), adapted
here to a single pre-computed 2-D embedding instead of refitting PCA from
raw multi-dimensional Ca coordinates each call:

```python
df = load_pose_clusters("rna_ds_dcl4_drb2_drb4")

plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4")                                    # pipeline's own hierarchical-RMSD clusters (default)
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", color_by="backend", cluster_method=None)  # no re-clustering, colour by backend instead
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", color_by="ranking_score", cluster_method=None)  # continuous colour scale

plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="gmm", n_components="auto")   # GMM auto: BIC-knee sweep k=1..12
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="gmm", n_components=3)        # GMM manual: fit GMM-3

plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto: Optuna/DBCV search
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="hdbscan", n_components="manual",
         hdbscan_min_cluster_size=15)

# Ablation: does the ensemble still separate into the same pose clusters without a given backend?
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", models={"alphafold3": False})
```

**Every `plot_pca` call that shows a real cluster partition (the pipeline's
own `cluster` column, or a fresh `gmm`/`hdbscan` fit -- not plain
`color_by="backend"/"ptm"/"iptm"/"ranking_score"`) automatically symlinks
that clustering's CIFs to disk**, mirrors
`../../NPF-ab-initio-modelling/ABCfold_NPF_pipeline`'s `_reannotate`:

```text
results/<complex>/reannotated/<method_tag>/cluster_<k>/*.cif   -- up to max_per_cluster=20 CIFs, randomly subsampled
results/<complex>/reannotated/<method_tag>/assignments.csv     -- every model in every cluster, with a `symlinked` column
```

`method_tag` is `"pipeline"` for the default hierarchical-RMSD clusters,
`"gmm_auto_k<k>"`/`"gmm_k<k>"` for GMM, `"hdbscan_auto"`/`"hdbscan_manual"`
for HDBSCAN -- so re-running with different parameters lands in its own
subdirectory rather than overwriting a previous exploration. Point
ChimeraX/PyMOL at a `cluster_<k>/` directory directly to load every
structure in that cluster at once. Pass `reannotate_clusters=False` to
skip the disk writes during quick iteration.

**Every point is one predicted model** (backend x seed x sample); hovering
shows backend/seed/sample_index/cluster/ptm/iptm/ranking_score/cif_path so
you can trace an interesting point straight back to its structure file
under `results/<complex>/`.

**Caveat worth watching for:** backends don't all contribute the same
number of frames (e.g. right now `rna_ds_dcl4_drb2_drb4` only has
AlphaFold3/OpenFold3/RosettaFold3 -- Boltz-2/Chai-1/Protenix all hit
hardware/architecture limits on this complex's size, see project memory /
session history), which can dominate a GMM or HDBSCAN fit by sheer point
count. `models=` ablation above is the way to check whether a cluster
assignment actually depends on that imbalance.

Unlike `ABCfold_NPF_pipeline`'s equivalent notebook, `discover_predictions()`
in `scripts/abcfold_backends.py` already deduplicates RosettaFold3's
`..._model.cif`/`..._model_fixed.cif` pair upstream (at
`scripts/compress_abcfold_metadata.py` time), so there's no
near-duplicate-frame caveat to repeat here.

**Prerequisite:** `workflows/postprocessing/Snakefile`'s `pose_cluster` rule
must have run for a complex before `load_pose_clusters(complex_name)` can
read its `pose_clusters.csv`.


In [11]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import optuna
from hdbscan.validity import validity_index
from kneed import KneeLocator
from sklearn.cluster import HDBSCAN
from sklearn.mixture import GaussianMixture

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one INFO line per trial is too noisy at n_trials=200

ROOT = Path("..")
RESULTS_ROOT = ROOT / "results"
ABCFOLD_ROOT = RESULTS_ROOT / "abcfold"

CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_SYMBOLS = {
    "alphafold3": "circle", "boltz": "square", "chai1": "diamond",
    "openfold3": "triangle-up", "protenix": "x", "rosettafold3": "cross",
}

HDBSCAN_MIN_SAMPLES_CANDIDATES = [3, 5, 10, 15, 20, 25, 30]
HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
HDBSCAN_CLUSTER_SELECTION_METHODS = ["eom", "leaf"]
HDBSCAN_METRICS = ["euclidean", "cityblock"]


def load_pose_clusters(complex_name: str) -> pd.DataFrame:
    """results/<complex>/pose_clusters.csv, as written by
    scripts/pose_cluster_anchor.py -- pc1/pc2 are that script's own PCA fit
    on the anchor-Kabsch-aligned, RMSF-trimmed partner-chain feature vector
    (see its module docstring for the full algorithm). This notebook does
    NOT refit PCA or redo the Kabsch alignment -- it explores the exact
    embedding the pipeline already computed and persisted, so what you see
    here always matches results/<complex>/pose_clusters_pca.svg."""
    path = RESULTS_ROOT / complex_name / "pose_clusters.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found -- run workflows/postprocessing/Snakefile's pose_cluster "
            f"rule for {complex_name} first (needs scripts/pose_cluster_anchor.py "
            "2026-08-24 or later -- earlier runs didn't persist pc1/pc2)."
        )
    df = pd.read_csv(path)
    if "pc1" not in df.columns:
        raise ValueError(
            f"{path} has no pc1/pc2 columns -- re-run pose_cluster_anchor.py for "
            f"{complex_name} to regenerate it with the current script version."
        )
    var_path = RESULTS_ROOT / complex_name / "pose_clusters_pca_variance.json"
    variance = json.loads(var_path.read_text()) if var_path.exists() else {}
    print(f"[{complex_name}] {len(df)} models, {df['backend'].value_counts().to_dict()}")
    if variance:
        print(f"[{complex_name}] PC1={variance['pc1_explained_variance']:.1%} "
              f"PC2={variance['pc2_explained_variance']:.1%} "
              f"(sum={variance['pc1_explained_variance'] + variance['pc2_explained_variance']:.1%})")
    return df


# ── GMM auto (BIC-knee sweep) ───────────────────────────────────────────────

def _fit_gmm_bic_sweep(xy, k_min=1, k_max=12, n_init=20, random_state=42):
    """Fit a GaussianMixture for every k in [k_min, k_max] and return the
    one sitting at the knee of the BIC-vs-k curve. KneeLocator's default
    interpolation follows every point exactly, so a single noisy BIC value
    reads as a spurious knee right at the first bump -- fitting a
    polynomial through the curve first smooths that out. Falls back to the
    raw BIC minimum if KneeLocator finds no knee. Same approach as
    ABCfold_NPF_pipeline/scripts/_notebook_setup_functions.py's
    _fit_gmm_bic_sweep."""
    k_max = min(k_max, xy.shape[0] - 1)
    ks = list(range(max(1, k_min), k_max + 1))

    gmms, bic_by_k = {}, {}
    for k in ks:
        gmm = GaussianMixture(n_components=k, covariance_type="full",
                               n_init=n_init, random_state=random_state)
        try:
            gmm.fit(xy)
        except ValueError as e:
            print(f"[gmm-auto] WARNING: k={k} failed ({e}), skipping")
            continue
        gmms[k] = gmm
        bic_by_k[k] = float(gmm.bic(xy))

    if not bic_by_k:
        raise RuntimeError(
            f"GMM auto (BIC sweep) failed for every k in [{ks[0]}, {ks[-1]}] -- "
            "try a narrower auto_k_min/auto_k_max range or n_components=<int> (manual)")
    ks = sorted(bic_by_k)
    best_k = ks[int(np.argmin([bic_by_k[k] for k in ks]))]
    if len(ks) >= 3:
        degree = min(7, max(1, len(ks) - 3))
        try:
            kl = KneeLocator(ks, [bic_by_k[k] for k in ks],
                              curve="convex", direction="decreasing",
                              interp_method="polynomial", polynomial_degree=degree)
            if kl.knee is not None:
                best_k = int(kl.knee)
        except Exception as e:
            print(f"[gmm-auto] WARNING: KneeLocator failed ({e}), falling back to BIC minimum")

    return gmms[best_k].predict(xy), best_k, bic_by_k


def _plot_bic_curve(bic_by_k, best_k, title):
    ks = sorted(bic_by_k)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ks, y=[bic_by_k[k] for k in ks], mode="lines+markers",
                              line=dict(color="#1565C0", width=2), marker=dict(size=6), name="BIC"))
    fig.add_trace(go.Scatter(x=[best_k], y=[bic_by_k[best_k]], mode="markers",
                              marker=dict(size=14, color="#d62728", symbol="star"), name=f"knee k={best_k}"))
    fig.update_layout(title=f"{title}<br>BIC sweep k={ks[0]}-{ks[-1]}, knee k={best_k}",
                       xaxis_title="n_components (k)", yaxis_title="BIC",
                       template="plotly_white", height=340, width=480, showlegend=False)
    fig.show()


# ── HDBSCAN auto (Optuna/DBCV search) ───────────────────────────────────────

def _fit_hdbscan_dbcv_search(xy, min_samples_candidates=HDBSCAN_MIN_SAMPLES_CANDIDATES,
                              min_cluster_size_candidates=HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES,
                              cluster_selection_methods=HDBSCAN_CLUSTER_SELECTION_METHODS,
                              metrics=HDBSCAN_METRICS, n_trials=200, random_state=42):
    """Optuna/TPE search over (min_samples, min_cluster_size,
    cluster_selection_method, metric) for HDBSCAN, scored by DBCV (Moulavi
    et al. 2014) via hdbscan.validity.validity_index. Combos yielding fewer
    than 2 clusters, or erroring inside DBCV, score -1.0 so they're never
    selected. Same approach as ABCfold_NPF_pipeline/scripts/
    _notebook_setup_functions.py's _fit_hdbscan_dbcv_search."""
    n = xy.shape[0]
    max_min_cluster_size = max(2, n // 5)
    candidate_min_cluster_sizes = [m for m in min_cluster_size_candidates if 2 <= m <= max_min_cluster_size]
    if not candidate_min_cluster_sizes:
        candidate_min_cluster_sizes = [max_min_cluster_size]

    grid_size = (len(min_samples_candidates) * len(candidate_min_cluster_sizes)
                 * len(cluster_selection_methods) * len(metrics))
    n_trials = min(n_trials, grid_size)

    def objective(trial):
        min_samples = trial.suggest_categorical("min_samples", list(min_samples_candidates))
        min_cluster_size = trial.suggest_categorical("min_cluster_size", candidate_min_cluster_sizes)
        cluster_selection_method = trial.suggest_categorical("cluster_selection_method", list(cluster_selection_methods))
        metric = trial.suggest_categorical("metric", list(metrics))
        try:
            labels = HDBSCAN(min_samples=min_samples, min_cluster_size=min_cluster_size,
                              cluster_selection_method=cluster_selection_method,
                              metric=metric, copy=False).fit(xy).labels_
            n_clust = len(set(c for c in labels if c >= 0))
            dbcv = float(validity_index(xy.astype(np.float64), labels, metric=metric)) if n_clust >= 2 else -1.0
        except Exception as e:
            print(f"[hdbscan-auto] combo ms={min_samples} mcs={min_cluster_size} "
                  f"{cluster_selection_method}/{metric} failed: {e}")
            labels, dbcv = None, -1.0
        trial.set_user_attr("labels", None if labels is None else labels.tolist())
        return dbcv

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_trial = study.best_trial
    best_labels = best_trial.user_attrs["labels"]
    best = {"min_samples": best_trial.params["min_samples"],
            "min_cluster_size": best_trial.params["min_cluster_size"],
            "cluster_selection_method": best_trial.params["cluster_selection_method"],
            "metric": best_trial.params["metric"], "dbcv": best_trial.value}
    labels = np.array(best_labels) if best_labels is not None else np.full(n, -1)
    return labels, best, study


def _plot_dbcv_search(study, best, title):
    dbcvs = sorted(study.trials_dataframe()["value"].fillna(-1.0).tolist(), reverse=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=list(range(len(dbcvs))), y=dbcvs, marker_color="#1565C0", name="DBCV"))
    fig.add_hline(y=best["dbcv"], line_dash="dash", line_color="#d62728",
                  annotation_text=f"best DBCV={best['dbcv']:.3f}")
    fig.update_layout(title=f"{title}<br>HDBSCAN Optuna/TPE search, {len(dbcvs)} trials",
                       xaxis_title="trial (sorted by DBCV)", yaxis_title="DBCV",
                       template="plotly_white", height=340, width=480, showlegend=False)
    fig.show()


# ── Reannotate: symlink each cluster's CIFs to disk ─────────────────────────

def reannotate(df: pd.DataFrame, complex_name: str, method_tag: str,
                label_col: str = "_color_label", max_per_cluster: int = 20,
                sample_seed: int = 42, out_dir: Path = None) -> Path:
    """Symlink each cluster's raw ABCfold CIFs (as referenced by df's
    cif_path, relative to results/abcfold/) into
    results/<complex>/reannotated/<method_tag>/cluster_<k>/, so you can
    point ChimeraX/PyMOL/etc. at a whole cluster directory directly instead
    of cross-referencing pose_clusters.csv by hand. Mirrors
    ../../NPF-ab-initio-modelling/ABCfold_NPF_pipeline/scripts/
    _notebook_setup_functions.py's _reannotate, simplified: this project's
    cif_path is already a single resolvable path per row (no apo/holo
    source_run split to disambiguate), and
    scripts/abcfold_backends.discover_predictions() already deduplicates
    RosettaFold3's raw/_fixed CIF pair upstream (at
    scripts/compress_abcfold_metadata.py time), so there's no collision
    case to skip here.

    Clusters routinely hold more structures than is useful to load at
    once, so at most max_per_cluster per cluster are randomly subsampled
    (without replacement, sample_seed for reproducibility) and only those
    get symlinked to disk -- assignments.csv still lists every row in the
    cluster (with a 'symlinked' column) so full membership stays available
    even though the on-disk CIF set is capped. Any stale symlinks from a
    previous call are removed first, so re-running with different
    clustering parameters doesn't leave orphaned files behind.

    Called automatically by plot_pca whenever it assigns real cluster
    labels (the pipeline's own 'cluster' column, or a fresh gmm/hdbscan
    fit) -- pass reannotate=False to plot_pca to skip it for quick
    iteration."""
    out_dir = (RESULTS_ROOT / complex_name / "reannotated" / method_tag) if out_dir is None else out_dir
    cluster_ids = sorted(df[label_col].unique(), key=str)

    assign_rows = []
    n_symlinked = 0
    for cid in cluster_ids:
        dir_name = str(cid).replace(" ", "_").replace("/", "_")
        cluster_dir = out_dir / dir_name
        cluster_dir.mkdir(parents=True, exist_ok=True)
        for stale in cluster_dir.iterdir():
            if stale.is_symlink():
                stale.unlink()

        cluster_rows = df[df[label_col] == cid]
        sampled_idx = set(cluster_rows.sample(
            n=min(len(cluster_rows), max_per_cluster), random_state=sample_seed,
        ).index)

        for idx, row in cluster_rows.iterrows():
            cif = ABCFOLD_ROOT / row["cif_path"]
            symlinked = idx in sampled_idx and cif.exists()
            if symlinked:
                dest = cluster_dir / f"{row['backend']}_seed{row['seed']}_sample{row['sample_index']}.cif"
                if not dest.exists():
                    dest.symlink_to(cif.resolve())
                    n_symlinked += 1
            assign_rows.append({
                "backend": row["backend"], "seed": row["seed"], "sample_index": row["sample_index"],
                "cluster": cid, "ptm": row["ptm"], "iptm": row["iptm"],
                "ranking_score": row["ranking_score"], "cif_path": row["cif_path"],
                "symlinked": symlinked,
            })
    pd.DataFrame(assign_rows).to_csv(out_dir / "assignments.csv", index=False)
    print(f"[reannotate] {complex_name}/{method_tag}: {n_symlinked} symlinks "
          f"(max {max_per_cluster}/cluster) of {len(assign_rows)} assignments -> {out_dir}")
    return out_dir


# ── Main plotting entry point ───────────────────────────────────────────────

def plot_pca(df: pd.DataFrame, complex_name: str = "", models: dict = None,
             color_by: str = "cluster", cluster_method: str = None, n_components=None,
             auto_k_min: int = 1, auto_k_max: int = 12,
             hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
             hdbscan_cluster_selection_method: str = "eom", hdbscan_metric: str = "euclidean",
             hdbscan_n_trials: int = 200,
             marker_size: int = 7, opacity: float = 0.75,
             reannotate_clusters: bool = True, max_per_cluster: int = 20):
    """Interactive Plotly scatter of one complex's pc1/pc2 (see
    load_pose_clusters -- this does NOT refit PCA, it plots the pipeline's
    own embedding).

    Parameters
    ----------
    df              output of load_pose_clusters(complex_name).
    models          ablation switch -- dict of backend name -> True/False;
                    only rows for backends mapped to True are plotted/
                    clustered. Defaults to every backend present. E.g.
                    models={"alphafold3": False} to ask whether the
                    remaining backends' ensemble still covers the same pose
                    clusters without AF3.
    color_by        "cluster" (default -- the pipeline's own hierarchical
                    RMSD clustering, ignored if cluster_method is set),
                    "backend", "ptm", "iptm", or "ranking_score".
    cluster_method  None (default -- use the existing 'cluster' column),
                    "gmm", or "hdbscan": re-cluster pc1/pc2 fresh with that
                    method instead, overriding color_by.
    n_components    "gmm": int fits exactly that many components (manual);
                    "auto" (default when cluster_method="gmm") sweeps
                    auto_k_min..auto_k_max and picks the BIC-vs-k knee
                    (see _fit_gmm_bic_sweep), with a BIC diagnostic plot.
                    "hdbscan": "auto" (default when cluster_method=
                    "hdbscan") searches hyperparameters via Optuna/TPE
                    scored by DBCV (see _fit_hdbscan_dbcv_search), with a
                    DBCV diagnostic plot; "manual" fits HDBSCAN directly
                    with the explicit hdbscan_* arguments below
                    (hdbscan_min_cluster_size then required). Points
                    HDBSCAN calls noise (-1) are shown as "noise".
    hdbscan_*       explicit HDBSCAN hyperparameters, only used when
                    cluster_method="hdbscan", n_components="manual".
    hdbscan_n_trials  Optuna trial budget for the "auto" HDBSCAN search.
    reannotate_clusters  when a real cluster partition is shown (pipeline's
                    own 'cluster' column, or a fresh gmm/hdbscan fit --
                    NOT for color_by="backend"/"ptm"/"iptm"/"ranking_score"),
                    symlink each cluster's CIFs into results/<complex>/
                    reannotated/<method_tag>/cluster_<k>/ (see reannotate()).
                    Default True, matching ABCfold_NPF_pipeline's notebooks;
                    set False to skip disk writes during quick iteration.
    max_per_cluster  cap on how many CIFs per cluster get symlinked when
                    reannotate_clusters fires (randomly subsampled).
    """
    if models is not None:
        keep = df["backend"].isin([b for b, use in models.items() if use])
        missing = set(models) - set(df["backend"].unique())
        if missing:
            print(f"[plot_pca] note: models={sorted(missing)} not present in this data, ignored")
        df = df.loc[keep].reset_index(drop=True)
        if df.empty:
            raise ValueError(f"models={models!r} leaves no rows -- enable at least one present backend")

    xy = df[["pc1", "pc2"]].to_numpy()
    title = f"{complex_name}: PCA (partner chains)" if complex_name else "PCA (partner chains)"

    if cluster_method == "gmm":
        k_mode = "auto" if n_components in (None, "auto") else n_components
        if k_mode == "auto":
            labels, best_k, bic_by_k = _fit_gmm_bic_sweep(xy, k_min=auto_k_min, k_max=auto_k_max)
            _plot_bic_curve(bic_by_k, best_k, title)
        else:
            gmm = GaussianMixture(n_components=int(k_mode), covariance_type="full", n_init=20, random_state=42)
            labels = gmm.fit_predict(xy)
        df = df.assign(_color_label=[f"gmm {label}" for label in labels])
        color_col, is_categorical = "_color_label", True
        method_tag = f"gmm_auto_k{best_k}" if k_mode == "auto" else f"gmm_k{k_mode}"
        if reannotate_clusters:
            reannotate(df, complex_name, method_tag, max_per_cluster=max_per_cluster)

    elif cluster_method == "hdbscan":
        mode = "auto" if n_components in (None, "auto") else n_components
        if mode == "auto":
            labels, best, study = _fit_hdbscan_dbcv_search(xy, n_trials=hdbscan_n_trials)
            print(f"[hdbscan-auto] best: {best}")
            _plot_dbcv_search(study, best, title)
        else:
            if hdbscan_min_cluster_size is None:
                raise ValueError('cluster_method="hdbscan", n_components="manual" needs hdbscan_min_cluster_size')
            labels = HDBSCAN(min_samples=hdbscan_min_samples, min_cluster_size=hdbscan_min_cluster_size,
                              cluster_selection_method=hdbscan_cluster_selection_method,
                              metric=hdbscan_metric).fit(xy).labels_
        df = df.assign(_color_label=["noise" if label < 0 else f"hdbscan {label}" for label in labels])
        color_col, is_categorical = "_color_label", True
        method_tag = "hdbscan_auto" if mode == "auto" else "hdbscan_manual"
        if reannotate_clusters:
            reannotate(df, complex_name, method_tag, max_per_cluster=max_per_cluster)

    elif color_by == "cluster":
        df = df.assign(_color_label=[f"cluster {c}" for c in df["cluster"]])
        color_col, is_categorical = "_color_label", True
        if reannotate_clusters:
            reannotate(df, complex_name, "pipeline", max_per_cluster=max_per_cluster)
    elif color_by == "backend":
        color_col, is_categorical = "backend", True
    elif color_by in ("ptm", "iptm", "ranking_score"):
        color_col, is_categorical = color_by, False
    else:
        raise ValueError(f"color_by={color_by!r} not recognized (cluster/backend/ptm/iptm/ranking_score)")

    hover_cols = ["backend", "seed", "sample_index", "cluster", "ptm", "iptm", "ranking_score", "cif_path"]
    fig = go.Figure()
    if is_categorical:
        categories = sorted(df[color_col].unique(), key=str)
        palette = CLUSTER_PALETTE if len(categories) <= len(CLUSTER_PALETTE) else px.colors.qualitative.Alphabet
        for i, cat in enumerate(categories):
            sub = df[df[color_col] == cat]
            for backend in sorted(sub["backend"].unique()):
                bsub = sub[sub["backend"] == backend]
                fig.add_trace(go.Scatter(
                    x=bsub["pc1"], y=bsub["pc2"], mode="markers",
                    marker=dict(size=marker_size, opacity=opacity, color=palette[i % len(palette)],
                                symbol=BACKEND_SYMBOLS.get(backend, "circle"),
                                line=dict(width=0.5, color="white")),
                    name=f"{cat} / {backend}",
                    customdata=bsub[hover_cols],
                    hovertemplate="<br>".join(f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)) + "<extra></extra>",
                ))
    else:
        fig.add_trace(go.Scatter(
            x=df["pc1"], y=df["pc2"], mode="markers",
            marker=dict(size=marker_size, opacity=opacity, color=df[color_col],
                        colorscale="Viridis", showscale=True, colorbar=dict(title=color_by),
                        line=dict(width=0.5, color="white")),
            customdata=df[hover_cols],
            hovertemplate="<br>".join(f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)) + "<extra></extra>",
        ))

    fig.update_layout(
        title=title, xaxis_title="PC1", yaxis_title="PC2",
        template="plotly_white", height=620, width=820,
        legend=dict(font=dict(size=9)),
    )
    fig.show()
    return df


## Load data

In [12]:
df = load_pose_clusters("rna_ds_dcl4_drb2_drb4")
df.head()

[rna_ds_dcl4_drb2_drb4] 300 models, {'openfold3': 100, 'rosettafold3': 100, 'alphafold3': 100}
[rna_ds_dcl4_drb2_drb4] PC1=45.2% PC2=12.5% (sum=57.7%)


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795


## Examples -- `rna_ds_dcl4_drb2_drb4`

One call per cell (not one cell with several `plot_pca` calls): each call
opens its own figure, so splitting them out lets you zoom/pan/hover one
plot for as long as you like without a later cell's output replacing it.
Edit a cell's arguments directly for one-off exploration without touching
any other cell.


In [13]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4")  # pipeline's own hierarchical-RMSD clusters

[reannotate] rna_ds_dcl4_drb2_drb4/pipeline: 39 symlinks (max 20/cluster) of 300 assignments -> ../results/rna_ds_dcl4_drb2_drb4/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474,cluster 1
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258,cluster 1
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910,cluster 1
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373,cluster 2
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795,cluster 1
...,...,...,...,...,...,...,...,...,...,...,...,...
295,alphafold3,9,0.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,2.058147e+00,485.726951,950.917292,cluster 1
296,alphafold3,9,1.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.440000,0.300000,0.330000,1,3.999441e+00,525.662680,464.385904,cluster 1
297,alphafold3,9,2.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.470000,0.290000,0.400000,1,1.969119e+00,499.478096,388.692787,cluster 1
298,alphafold3,9,3.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,1.740857e+00,518.441257,-405.500169,cluster 1


In [14]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", color_by="backend", cluster_method=None)  # no re-clustering, coloured by backend

,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795
...,...,...,...,...,...,...,...,...,...,...,...
295,alphafold3,9,0.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,2.058147e+00,485.726951,950.917292
296,alphafold3,9,1.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.440000,0.300000,0.330000,1,3.999441e+00,525.662680,464.385904
297,alphafold3,9,2.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.470000,0.290000,0.400000,1,1.969119e+00,499.478096,388.692787
298,alphafold3,9,3.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,1.740857e+00,518.441257,-405.500169


In [15]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)

[reannotate] rna_ds_dcl4_drb2_drb4/gmm_auto_k3: 56 symlinks (max 20/cluster) of 300 assignments -> ../results/rna_ds_dcl4_drb2_drb4/reannotated/gmm_auto_k3


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474,gmm 2
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258,gmm 2
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910,gmm 2
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373,gmm 1
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795,gmm 2
...,...,...,...,...,...,...,...,...,...,...,...,...
295,alphafold3,9,0.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,2.058147e+00,485.726951,950.917292,gmm 0
296,alphafold3,9,1.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.440000,0.300000,0.330000,1,3.999441e+00,525.662680,464.385904,gmm 0
297,alphafold3,9,2.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.470000,0.290000,0.400000,1,1.969119e+00,499.478096,388.692787,gmm 0
298,alphafold3,9,3.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,1.740857e+00,518.441257,-405.500169,gmm 0


In [18]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="gmm", n_components=4) 

[reannotate] rna_ds_dcl4_drb2_drb4/gmm_k4: 75 symlinks (max 20/cluster) of 300 assignments -> ../results/rna_ds_dcl4_drb2_drb4/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474,gmm 1
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258,gmm 1
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910,gmm 1
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373,gmm 0
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795,gmm 1
...,...,...,...,...,...,...,...,...,...,...,...,...
295,alphafold3,9,0.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,2.058147e+00,485.726951,950.917292,gmm 2
296,alphafold3,9,1.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.440000,0.300000,0.330000,1,3.999441e+00,525.662680,464.385904,gmm 2
297,alphafold3,9,2.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.470000,0.290000,0.400000,1,1.969119e+00,499.478096,388.692787,gmm 2
298,alphafold3,9,3.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,1.740857e+00,518.441257,-405.500169,gmm 2


In [16]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)

[hdbscan-auto] best: {'min_samples': 20, 'min_cluster_size': 40, 'cluster_selection_method': 'eom', 'metric': 'euclidean', 'dbcv': 0.6482846758720047}


[reannotate] rna_ds_dcl4_drb2_drb4/hdbscan_auto: 60 symlinks (max 20/cluster) of 300 assignments -> ../results/rna_ds_dcl4_drb2_drb4/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474,hdbscan 0
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258,hdbscan 0
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910,hdbscan 0
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373,hdbscan 1
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795,hdbscan 0
...,...,...,...,...,...,...,...,...,...,...,...,...
295,alphafold3,9,0.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,2.058147e+00,485.726951,950.917292,noise
296,alphafold3,9,1.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.440000,0.300000,0.330000,1,3.999441e+00,525.662680,464.385904,hdbscan 0
297,alphafold3,9,2.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.470000,0.290000,0.400000,1,1.969119e+00,499.478096,388.692787,hdbscan 0
298,alphafold3,9,3.0,rna_ds_dcl4_drb2_drb4/alphafold3_rna_ds_dcl4_d...,0.480000,0.300000,0.340000,1,1.740857e+00,518.441257,-405.500169,hdbscan 0


In [17]:
plot_pca(df, complex_name="rna_ds_dcl4_drb2_drb4", models={**{b: True for b in df["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded

[reannotate] rna_ds_dcl4_drb2_drb4/pipeline: 36 symlinks (max 20/cluster) of 200 assignments -> ../results/rna_ds_dcl4_drb2_drb4/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490186,0.264896,0.312163,1,9.090180e-07,390.243859,-195.880474,cluster 1
1,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.480115,0.247478,0.296615,1,1.632735e+00,327.892915,-331.262258,cluster 1
2,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.495057,0.255009,0.316674,1,1.697513e+00,515.260323,-198.263910,cluster 1
3,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.490275,0.265788,0.317111,2,2.206372e+00,-660.094679,459.128373,cluster 2
4,openfold3,1,NaN,rna_ds_dcl4_drb2_drb4/openfold_rna_ds_dcl4_drb...,0.472669,0.247302,0.302617,1,1.736471e+00,315.007277,-272.111795,cluster 1
...,...,...,...,...,...,...,...,...,...,...,...,...
195,rosettafold3,9,0.0,rna_ds_dcl4_drb2_drb4/rosettafold_rna_ds_dcl4_...,0.445602,0.355604,-99.626404,2,3.908835e+00,-563.928572,-248.366451,cluster 2
196,rosettafold3,9,1.0,rna_ds_dcl4_drb2_drb4/rosettafold_rna_ds_dcl4_...,0.445663,0.355405,-99.626503,2,4.663759e+00,-266.312025,-165.190042,cluster 2
197,rosettafold3,9,2.0,rna_ds_dcl4_drb2_drb4/rosettafold_rna_ds_dcl4_...,0.441403,0.350983,-99.630898,1,4.988918e+00,297.655151,472.049284,cluster 1
198,rosettafold3,9,3.0,rna_ds_dcl4_drb2_drb4/rosettafold_rna_ds_dcl4_...,0.446065,0.355547,-99.626297,1,3.930547e+00,-349.636714,-214.359439,cluster 1
